In [2]:
import logging
from pathlib import Path

import numpy as np
import pandas as pd

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# Paths — notebook için direkt path
DATA_DIR = Path("../data")
RAW_FILE = DATA_DIR / "raw" / "funds_raw.parquet"
FEATURES_FILE = DATA_DIR / "processed" / "funds_features.parquet"

print(RAW_FILE)
print(RAW_FILE.exists())

../data/raw/funds_raw.parquet
True


In [3]:
df = pd.read_parquet(RAW_FILE)
print(df.shape)
print(df.dtypes)
df.head()

(705783, 6)
date               object
code               object
title              object
price             float64
category_rank       int64
category_total      int64
dtype: object


,date,code,title,price,category_rank,category_total
0,2025-09-25,AU1,A1 CAPİTAL PORTFÖY ALTIN FONU,0.0,0,28
1,2025-09-26,AU1,A1 CAPİTAL PORTFÖY ALTIN FONU,0.0,0,28
2,2025-09-29,AU1,A1 CAPİTAL PORTFÖY ALTIN FONU,0.0,0,28
3,2025-09-30,AU1,A1 CAPİTAL PORTFÖY ALTIN FONU,0.0,0,28
4,2025-10-01,AU1,A1 CAPİTAL PORTFÖY ALTIN FONU,0.0,0,28


In [4]:
# Tarih sütununu düzelt
df["date"] = pd.to_datetime(df["date"])

# Price sıfır olan satırları temizle
df = df[df["price"] > 0]

# Sırala
df = df.sort_values(["code", "date"]).reset_index(drop=True)

print(df.shape)
print(df.dtypes)
df.head()

(686115, 6)
date              datetime64[ns]
code                      object
title                     object
price                    float64
category_rank              int64
category_total             int64
dtype: object


,date,code,title,price,category_rank,category_total
0,2021-05-18,AAL,ATA PORTFÖY PARA PİYASASI (TL) FONU,0.673464,22,83
1,2021-05-20,AAL,ATA PORTFÖY PARA PİYASASI (TL) FONU,0.674095,22,83
2,2021-05-21,AAL,ATA PORTFÖY PARA PİYASASI (TL) FONU,0.674445,22,83
3,2021-05-24,AAL,ATA PORTFÖY PARA PİYASASI (TL) FONU,0.675385,22,83
4,2021-05-25,AAL,ATA PORTFÖY PARA PİYASASI (TL) FONU,0.675741,22,83


## Feature Eng

In [5]:
# daily return
df["daily_return"] = df.groupby("code")["price"].pct_change()

# control
print(df[["code", "date", "price", "daily_return"]].head(10))

  code       date     price  daily_return
0  AAL 2021-05-18  0.673464           NaN
1  AAL 2021-05-20  0.674095      0.000937
2  AAL 2021-05-21  0.674445      0.000519
3  AAL 2021-05-24  0.675385      0.001394
4  AAL 2021-05-25  0.675741      0.000527
5  AAL 2021-05-26  0.676070      0.000487
6  AAL 2021-05-27  0.676393      0.000478
7  AAL 2021-05-28  0.676716      0.000478
8  AAL 2021-05-31  0.677638      0.001362
9  AAL 2021-06-01  0.677996      0.000528


In [7]:
def compute_returns(group):
    """Compute 1Y, 2Y, 4Y returns for a single fund."""
    today = group["date"].max()
    price_today = group[group["date"] == today]["price"].values[0]
    
    results = {}
    for years, key in [(1, "return_1y"), (2, "return_2y"), (4, "return_4y")]:
        cutoff = today - pd.DateOffset(years=years)
        past_prices = group[group["date"] <= cutoff]["price"]
        
        if past_prices.empty:
            results[key] = np.nan
        else:
            results[key] = (price_today - past_prices.iloc[-1]) / past_prices.iloc[-1]
    
    return pd.Series(results)

returns_df = df.groupby("code").apply(compute_returns).reset_index()
print(returns_df.head(10))

  code  return_1y  return_2y  return_4y
0  AAL   0.504094   1.401126   3.113493
1  AAS   0.508284   0.709205   2.612077
2  AAV   0.345852   0.291697   7.186247
3  AC1   0.466016   1.222186        NaN
4  AC4   0.526396   1.439383        NaN
5  AC5   0.412007   0.784634        NaN
6  AC6   0.224947   0.510793        NaN
7  ADE   0.425413   1.211141   2.867255
8  ADP   0.396507   0.270797   8.339342
9  AED   0.469879   0.619458   5.477809


/var/folders/81/c9rwyrnx0pn6ykwcftvgylkr0000gn/T/ipykernel_15151/3563065430.py:18: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  returns_df = df.groupby("code").apply(compute_returns).reset_index()


In [8]:
def compute_risk_metrics(group):
    """Compute volatility, Sharpe, Sortino, max drawdown."""
    returns = group["daily_return"].dropna()
    
    if len(returns) < 30:
        return pd.Series({
            "volatility": np.nan,
            "sharpe": np.nan,
            "sortino": np.nan,
            "max_drawdown": np.nan
        })
    
    # Annualized volatility
    volatility = returns.std() * np.sqrt(252)
    
    # Annualized Sharpe (risk-free = 0)
    sharpe = (returns.mean() * 252) / volatility if volatility > 0 else np.nan
    
    # Sortino — only downside volatility
    downside = returns[returns < 0].std() * np.sqrt(252)
    sortino = (returns.mean() * 252) / downside if downside > 0 else np.nan
    
    # Max drawdown
    cumulative = (1 + returns).cumprod()
    rolling_max = cumulative.cummax()
    drawdown = (cumulative - rolling_max) / rolling_max
    max_drawdown = drawdown.min()
    
    return pd.Series({
        "volatility": volatility,
        "sharpe": sharpe,
        "sortino": sortino,
        "max_drawdown": max_drawdown
    })

risk_df = df.groupby("code").apply(compute_risk_metrics).reset_index()
print(risk_df.head(10))

  code  volatility     sharpe    sortino  max_drawdown
0  AAL    0.015773  20.036196        NaN      0.000000
1  AAS    0.143815   1.832358   2.372080     -0.142407
2  AAV    0.269629   2.095055   2.739347     -0.267733
3  AC1    0.015394  24.884522        NaN      0.000000
4  AC4    0.017721  24.196461  12.435108     -0.003475
5  AC5    0.109479   3.366323   2.119118     -0.098242
6  AC6    0.092932   3.738872   5.499098     -0.046960
7  ADE    0.036940   8.150863   5.628031     -0.058551
8  ADP    0.401773   1.565712   2.401066     -0.366855
9  AED    0.174849   2.728380   3.502973     -0.167690


/var/folders/81/c9rwyrnx0pn6ykwcftvgylkr0000gn/T/ipykernel_15151/1771028952.py:36: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  risk_df = df.groupby("code").apply(compute_risk_metrics).reset_index()


In [9]:
# Merge returns and risk metrics
features_df = returns_df.merge(risk_df, on="code")

# Add fund title
title_map = df.groupby("code")["title"].last()
features_df = features_df.merge(title_map, on="code")

print(features_df.shape)
print(features_df.head())

(900, 9)
  code  return_1y  return_2y  return_4y  volatility     sharpe    sortino  \
0  AAL   0.504094   1.401126   3.113493    0.015773  20.036196        NaN   
1  AAS   0.508284   0.709205   2.612077    0.143815   1.832358   2.372080   
2  AAV   0.345852   0.291697   7.186247    0.269629   2.095055   2.739347   
3  AC1   0.466016   1.222186        NaN    0.015394  24.884522        NaN   
4  AC4   0.526396   1.439383        NaN    0.017721  24.196461  12.435108   

   max_drawdown                                              title  
0      0.000000                ATA PORTFÖY PARA PİYASASI (TL) FONU  
1     -0.142407                ATA PORTFÖY FON SEPETİ SERBEST FONU  
2     -0.267733  ATA PORTFÖY İKİNCİ HİSSE SENEDİ (TL) FONU (HİS...  
3      0.000000    PARDUS PORTFÖY KISA VADELİ KATILIM SERBEST FONU  
4     -0.003475              PARDUS PORTFÖY PARA PİYASASI (TL) FON  


In [10]:
# Save to processed folder
FEATURES_FILE.parent.mkdir(parents=True, exist_ok=True)
features_df.to_parquet(FEATURES_FILE, index=False)
print(f"Saved: {FEATURES_FILE}")
print(f"Shape: {features_df.shape}")

Saved: ../data/processed/funds_features.parquet
Shape: (900, 9)


In [2]:
import pandas as pd

RAW_FILE = "/Users/ulasmerttoy/tefas-assistant/data/raw/funds_raw.parquet"
df = pd.read_parquet(RAW_FILE)

print("Satır sayısı:", len(df))
print("\nSütunlar:")
print(df.columns.tolist())

Satır sayısı: 705783

Sütunlar:
['date', 'code', 'title', 'price', 'category_rank', 'category_total']


In [5]:
import pandas as pd

feat = pd.read_parquet("/Users/ulasmerttoy/tefas-assistant/data/processed/funds_features.parquet")

print(feat["volatility"].describe())
print("\nYüzdelikler:")
print(feat["volatility"].quantile([0.33, 0.50, 0.66, 0.90]))

count    894.000000
mean       0.472228
std        9.401290
min        0.012886
25%        0.053947
50%        0.155277
75%        0.235107
max      281.229669
Name: volatility, dtype: float64

Yüzdelikler:
0.33    0.084186
0.50    0.155277
0.66    0.206981
0.90    0.280300
Name: volatility, dtype: float64


In [6]:
import pandas as pd

feat = pd.read_parquet("/Users/ulasmerttoy/tefas-assistant/data/processed/funds_features.parquet")

# Volatilitesi en yüksek 10 fon
print(feat.sort_values("volatility", ascending=False)[
    ["code", "title", "volatility", "return_1y", "sharpe"]
].head(10))

    code                                              title  volatility  \
534  NBH     NEO PORTFÖY ALGORİTMİK STRATEJİLER SERBEST FON  281.229669   
57   BFS                BULLS PORTFÖY EMTİA FON SEPETİ FONU    1.335898   
827  YCK            YAPI KREDİ PORTFÖY CİHANGİR SERBEST FON    0.985526   
744  SKZ                  NEO PORTFÖY SEKİZİNCİ SERBEST FON    0.789835   
72   BMU  BULLS PORTFÖY MUTLAK GETİRİ HEDEFLİ HİSSE SENE...    0.764114   
777  THV  ATA PORTFÖY BARBAROS HİSSE SENEDİ SERBEST (TL)...    0.728190   
312  GUK          GARANTİ PORTFÖY GÜMÜŞ KATILIM SERBEST FON    0.701880   
275  GMI                       AK PORTFÖY GÜMÜŞ SERBEST FON    0.657390   
189  ESP                     AURA PORTFÖY EMTİA SERBEST FON    0.640118   
432  KID          TRIVE PORTFÖY BİRİNCİ SERBEST (DÖVİZ) FON    0.601998   

     return_1y    sharpe  
534   0.611758  0.454261  
57         NaN  0.301006  
827   0.572778  0.154464  
744        NaN -0.220588  
72         NaN  1.330108  
777   3.0537

In [7]:
import pandas as pd

raw = pd.read_parquet("/Users/ulasmerttoy/tefas-assistant/data/raw/funds_raw.parquet")

nbh = raw[raw["code"] == "NBH"].sort_values("date")
print(nbh[["date", "price"]].describe())
print("\nEn düşük/yüksek fiyatlar:")
print(nbh.nsmallest(3, "price")[["date", "price"]])
print(nbh.nlargest(3, "price")[["date", "price"]])

             price
count  1256.000000
mean      2.525828
std       1.511491
min       0.000000
25%       1.144543
50%       2.400854
75%       3.609036
max       6.023744

En düşük/yüksek fiyatlar:
              date  price
426463  2021-05-18    0.0
426464  2021-05-20    0.0
426465  2021-05-21    0.0
              date     price
427718  2026-05-18  6.023744
427717  2026-05-15  6.019801
427716  2026-05-14  6.007389
